# 🔬 When Image-Quality Metrics Fail: Focus, Noise, Saturation, and Illumination

No-reference sharpness scores, tested on a synthetic acquisition where one nuisance changes at a time.

**Hardware:** CPU only. A free Colab CPU runtime is enough. This notebook does not check for a GPU, because it never uses one.

**Frameworks:** NumPy, SciPy, scikit-image, and Matplotlib. scikit-learn is imported for a single ROC-AUC in the last experiment.

**Prerequisites:** NumPy arrays, a Gaussian blur, and a 2-D power spectrum. Phase 01, [Frequency filtering](../../ComputationalSignalBase/FrequencyFiltering/image_filtering.ipynb), is sufficient. PyTorch is not required.

**Disclaimer:** For research and educational purposes only, not to be used for clinical diagnosis. Every image in this notebook is drawn by the code below. These are synthetic test patterns with hard-edged ellipses, not experimental microscopy, and a threshold fit here is not a threshold for a real microscope.

<a id="contents"></a>

## What you will learn

1. Separate a sharpness score, which tries to rank focus, from a gate such as the saturated-pixel fraction.
2. Check that four common scores fall as defocus increases, on a fixed object.
3. See which of those scores are moved by photon noise, object density, clipping, a dim exposure, or a vignette, with focus held fixed.
4. Fit a reject/keep cutoff on one set of fields and count false positives and false negatives on fields that were held out.

## Contents

1. [Setup](#setup)
2. [A synthetic acquisition](#acquisition)
3. [Scores and a saturation gate](#scores)
4. [Experiment 0 — Defocus, the positive control](#exp-defocus)
5. [Experiment 1 — Photon noise](#exp-noise)
6. [Experiment 2 — How much specimen is in the field](#exp-content)
7. [Experiment 3 — Clipping](#exp-clip)
8. [Experiment 4 — Vignetting and exposure](#exp-exposure)
9. [Where the scores disagree](#summary)
10. [A cutoff, with fields held out](#decision)
11. [Takeaways](#takeaways)
12. [Questions and exercises](#exercises)
13. [References](#references)


<a id="setup"></a>

## 1. Setup

The install cell does nothing if the packages are already importable, so a local environment is not upgraded. Colab already provides these packages; the cell is there for a clean runtime.


In [ ]:
# ── Environment ──
import importlib.util
import subprocess
import sys

_REQUIRED = {
    "numpy": "numpy",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}
_missing = [pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])


In [ ]:
# ── Imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from scipy.stats import spearmanr
from skimage.filters import laplace
from skimage.measure import blur_effect
from sklearn.metrics import roc_auc_score

SEED = 42

# Image and acquisition constants. Experiments below change one
# argument of acquire() at a time and leave the rest at these values.
SIZE = 192
IN_FOCUS_SIGMA = 0.7
PHOTONS = 800
READ_NOISE = 0.008

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})


def make_rng(*parts):
    """Deterministic generator from SEED and integer ids.

    Drawing numbers in one cell does not change the stream in another.
    """
    return np.random.default_rng(np.random.SeedSequence([SEED, *parts]))


def spearman(x, y):
    """Spearman rank correlation as a float, on older and newer SciPy."""
    result = spearmanr(x, y)
    if hasattr(result, "statistic"):
        return float(result.statistic)
    return float(result[0])


<a id="acquisition"></a>

## 2. A synthetic acquisition

The object $O$ is a stack of filled ellipses on a black background, with speckles inside the ellipses so the sharp pattern has high-frequency structure. It is not a cell and not a micrograph.

An exposure is

$$
\begin{aligned}
\sigma &= \sqrt{\sigma_0^2 + d^2}, \\
S &= \mathrm{clip}_{\geq 0}\big(g\, V (G_\sigma * O)\big), \\
I &= \mathrm{clip}_{[0,1]}\big(\mathrm{Poisson}(S \cdot n_{\mathrm{ph}})/n_{\mathrm{ph}} + \varepsilon\big),
\end{aligned}
$$

with $\varepsilon \sim \mathcal{N}(0, \sigma_r^2)$. $G_\sigma$ is a Gaussian of standard deviation $\sigma$ pixels, $\sigma_0$ is the in-focus blur, and $d$ is extra defocus in the same units. $V$ is 1 at the center and $1 - v$ at the corners. $g$ is a gain applied before the photon draw and the final clip, so $g > 1$ both brightens the image and flattens highlights. $n_{\mathrm{ph}}$ is the photon scale: lower means noisier.

This leaves out depth-dependent blur, a measured camera calibration, motion, and optics that are not a Gaussian. Those omissions are the reason the figures are labeled as simulations.


In [ ]:
# ── Synthetic object and exposure ──
def make_object(field, n_cells=40, size=SIZE):
    """Hard-edged ellipses with internal speckles.

    Parameters
    ----------
    field : int
        Seed id. The same id always rebuilds the same pattern.
    n_cells : int
        How many ellipses to draw. They may overlap.
    size : int
        Side length in pixels.

    Returns
    -------
    ndarray of float, shape (size, size), values in [0, 1].
    """
    rng = make_rng(1, int(field), int(n_cells), int(size))
    image = np.zeros((size, size), dtype=np.float64)
    yy, xx = np.indices((size, size))
    for _ in range(n_cells):
        cy, cx = rng.uniform(10, size - 10, size=2)
        ry, rx = rng.uniform(4.0, 9.0, size=2)
        angle = rng.uniform(0, np.pi)
        dy, dx = yy - cy, xx - cx
        yr = dy * np.cos(angle) + dx * np.sin(angle)
        xr = -dy * np.sin(angle) + dx * np.cos(angle)
        mask = (yr / ry) ** 2 + (xr / rx) ** 2 <= 1.0
        if not np.any(mask):
            continue
        base = rng.uniform(0.55, 0.9)
        speckles = rng.random(int(np.count_nonzero(mask))) > 0.55
        image[mask] = np.maximum(image[mask], base + 0.45 * speckles)
    return np.clip(image, 0.0, 1.0)


def acquire(
    obj,
    defocus=0.0,
    photons=PHOTONS,
    read_noise=READ_NOISE,
    gain=1.0,
    vignette=0.0,
    rng=None,
):
    """One grayscale exposure of a synthetic object. See the equation above.

    `rng` is required so each call has an explicit seed.
    """
    if rng is None:
        raise ValueError("Pass an explicit rng from make_rng().")
    sigma = float(np.sqrt(IN_FOCUS_SIGMA ** 2 + float(defocus) ** 2))
    blurred = ndimage.gaussian_filter(obj, sigma, mode="reflect")
    n = obj.shape[0]
    yy, xx = np.indices(obj.shape)
    radius = np.hypot(yy - (n - 1) / 2.0, xx - (n - 1) / 2.0)
    radius = radius / radius.max()
    signal = blurred * (1.0 - vignette * radius) * gain
    counts = rng.poisson(np.clip(signal, 0.0, None) * photons)
    image = counts / photons + rng.normal(0.0, read_noise, obj.shape)
    return np.clip(image, 0.0, 1.0)


# ── Five exposures of one object ──
_object = make_object(0)
_panels = [
    ("in focus", acquire(_object, rng=make_rng(2, 0))),
    ("defocus 2", acquire(_object, defocus=2.0, rng=make_rng(2, 1))),
    ("few photons", acquire(_object, photons=15, rng=make_rng(2, 2))),
    ("clipped", acquire(_object, gain=4.0, rng=make_rng(2, 3))),
    ("vignette", acquire(_object, vignette=0.85, rng=make_rng(2, 4))),
]
_fig, _axes = plt.subplots(1, len(_panels), figsize=(11, 2.5))
for _ax, (_title, _image) in zip(_axes, _panels):
    _ax.imshow(_image, cmap="gray", vmin=0, vmax=1)
    _ax.set_title(_title, fontsize=10)
    _ax.set_axis_off()
_fig.suptitle("One synthetic object under five exposures", fontsize=12)
_fig.tight_layout()
plt.show()


<a id="scores"></a>

## 3. Scores and a saturation gate

Each sharpness score is oriented so that a **larger value means sharper** on the defocus series in Experiment 0. That orientation is checked there; it is not assumed for the other experiments.

**Normalized variance**, $\mathrm{Var}(I) / \mathrm{mean}(I)$. Along a z-stack of one field, brightness and the amount of specimen stay roughly fixed, and this ratio ranks focus well (Sun, Duthaler, and Nelson, 2004). Across different fields those two things are not fixed.

**Laplacian variance**, the variance of the 3×3 Laplacian response from `skimage.filters.laplace`. It responds to high-frequency energy, so photon noise raises it. Pech-Pacheco et al. (2000) scored focus with the variance of the *absolute* Laplacian. The function below is the variance of the signed response, which is the usual short implementation, not their formula.

**Blur sharpness**, $1 - \texttt{blur\_effect}(I)$. Crete and coauthors (2007) score blur by how much a second blur still changes the image: a picture that is already blurred barely changes. `skimage.measure.blur_effect` implements that idea and returns a value near 0 for a sharp image and near 1 for a very blurred one, so we subtract it from 1. The re-blur window (`h_size`, default 11) has to stay fixed across images.

**Spectral slope**, the slope of $\log(\text{radial power})$ against $\log k$ for frequency indices $k = 3, \ldots, n/8$. A blur removes high frequencies, so this slope becomes more negative and the number we store gets smaller. The band stops at $n/8$ because, past that, a strong blur has already hit the read-noise floor and the fitted slope turns back up. This is a teaching fit. It is not CellProfiler's `PowerLogLogSlope`, and it is not the measurement in Field (1987); Field is why a log-log spectrum is a reasonable thing to look at.

**Saturated fraction**, the fraction of pixels at or above 0.99. This is a gate for clipping, not a sharpness score.

Peak signal-to-noise ratio and structural similarity need the unperturbed image. A first-pass screen does not have one. Both are used as reference metrics in the [bioimage denoising](../BioimageDenoising/self_supervised_bioimage_denoising.ipynb) notebook, and they are not recomputed here.


In [ ]:
# ── Sharpness scores and the saturation gate ──
def normalized_variance(image):
    """Variance divided by the mean. Larger means sharper on a fixed object."""
    return float(image.var() / max(float(image.mean()), 1e-8))


def laplacian_variance(image):
    """Variance of the Laplacian response. Larger means more high-frequency energy."""
    return float(laplace(image).var())


def blur_sharpness(image):
    """One minus scikit-image's no-reference blur effect. Larger means sharper."""
    return float(1.0 - blur_effect(image))


def spectral_slope(image, k_min=3):
    """Log-log slope of the radial power spectrum from k_min to n/8.

    The fit is ordinary least squares on the mean power in each integer
    frequency shell. A more negative slope is a blurrier image in
    Experiment 0, so the returned number is larger when the image is sharper.
    """
    centered = image - image.mean()
    power = np.abs(np.fft.fftshift(np.fft.fft2(centered))) ** 2
    n = image.shape[0]
    yy, xx = np.indices(image.shape)
    radius = np.hypot(yy - n // 2, xx - n // 2)
    k_max = max(k_min + 2, n // 8)
    shells = np.floor(radius).astype(np.int32)
    sums = np.bincount(shells.ravel(), power.ravel())
    counts = np.bincount(shells.ravel())
    ks = np.arange(k_min, k_max)
    profile = sums[ks] / np.maximum(counts[ks], 1)
    slope = np.polyfit(np.log(ks.astype(float)), np.log(profile), 1)[0]
    return float(slope)


def saturated_fraction(image, level=0.99):
    """Fraction of pixels at or above `level` after clipping to [0, 1]."""
    return float(np.mean(image >= level))


def variance_over_mean_squared(image):
    """Variance divided by the squared mean. Invariant to a global rescale
    when there is no additive noise and no clipping."""
    mean = max(float(image.mean()), 1e-8)
    return float(image.var() / mean ** 2)


def flat_field(image):
    """Divide by a heavy Gaussian estimate of the illumination."""
    background = ndimage.gaussian_filter(image, image.shape[0] / 8.0, mode="reflect")
    return image / np.maximum(background, 1e-3)


METRICS = {
    "normalized variance": normalized_variance,
    "Laplacian variance": laplacian_variance,
    "blur sharpness": blur_sharpness,
    "spectral slope": spectral_slope,
}


<a id="exp-defocus"></a>

## 4. Experiment 0 — Defocus, the positive control

**Question.** If the object, the exposure, and the noise draw's seed family stay fixed, does each score fall as defocus increases?

**Prediction.** All four should. If one does not, it is not usable as a focus score even before the nuisances.

Five objects are each exposed at seven defocus values. The curve is the mean across objects, and the band is ±1 standard deviation. The rank correlation is computed on the seven means, not on the individual fields.


In [ ]:
# ── Experiment 0: defocus sweep ──
DEFOCUS = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0])
N_CONTROL = 5

control = {name: np.empty((len(DEFOCUS), N_CONTROL)) for name in METRICS}
for d_index, defocus in enumerate(DEFOCUS):
    for field in range(N_CONTROL):
        image = acquire(
            make_object(field),
            defocus=float(defocus),
            rng=make_rng(10, field, d_index),
        )
        for name, score in METRICS.items():
            control[name][d_index, field] = score(image)

_fig, _axes = plt.subplots(1, 4, figsize=(12, 3.2), sharex=True)
for _ax, (name, values) in zip(_axes, control.items()):
    mean = values.mean(axis=1)
    std = values.std(axis=1)
    rho = spearman(DEFOCUS, mean)
    _ax.fill_between(DEFOCUS, mean - std, mean + std, alpha=0.25, color="#0072B2")
    _ax.plot(DEFOCUS, mean, marker="o", color="#0072B2")
    _ax.set_title(f"{name}\nSpearman {rho:.2f}", fontsize=10)
    _ax.set_xlabel("defocus $d$")
    assert rho <= -0.9, (name, rho)
_axes[0].set_ylabel("score (higher = sharper)")
_fig.tight_layout()
plt.show()
print("Positive control passed: every field-averaged score has Spearman ≤ -0.9 with defocus.")


**What this shows.** On a fixed object and a clean exposure, all four scores rank defocus correctly: the field-averaged Spearman correlation with $d$ is at most $-0.9$, which the cell asserts. The band is the spread across five objects. That spread is the warning for the next experiments. A score can separate focus levels of one object and still be shifted, by more than one focus step, when the object or the exposure changes.

The spectral slope only stays monotone because the fit stops at $k = n/8$. The exercise at the end asks what happens if that limit is raised.


<a id="exp-noise"></a>

## 5. Experiment 1 — Photon noise

**Question.** Holding the object fixed, can a defocused exposure with very few photons outscore a clean in-focus exposure?

**Prediction.** Laplacian variance should be the first to break, because it treats photon noise as edges. Blur sharpness compares an image with a re-blurred copy, so additive noise in both copies need not reverse the ranking.


In [ ]:
# ── Experiment 1: photon noise at two focus settings ──
PHOTON_GRID = np.array([2000, 400, 100, 40, 15])
N_NOISE = 6

noise = {
    name: {"sharp": np.empty((len(PHOTON_GRID), N_NOISE)),
           "defocused": np.empty((len(PHOTON_GRID), N_NOISE))}
    for name in METRICS
}
for p_index, photons in enumerate(PHOTON_GRID):
    for field in range(N_NOISE):
        obj = make_object(field)
        sharp = acquire(obj, photons=int(photons), rng=make_rng(11, field, p_index, 0))
        soft = acquire(
            obj, defocus=2.0, photons=int(photons), rng=make_rng(11, field, p_index, 1)
        )
        for name, score in METRICS.items():
            noise[name]["sharp"][p_index, field] = score(sharp)
            noise[name]["defocused"][p_index, field] = score(soft)

# Clean in-focus reference: the default photon scale, same six objects.
clean_lap = []
clean_blur = []
for field in range(N_NOISE):
    image = acquire(make_object(field), rng=make_rng(11, field, 99))
    clean_lap.append(laplacian_variance(image))
    clean_blur.append(blur_sharpness(image))
clean_lap = np.asarray(clean_lap)
clean_blur = np.asarray(clean_blur)

low = PHOTON_GRID.tolist().index(15)
lap_defocus_low = noise["Laplacian variance"]["defocused"][low]
blur_defocus_low = noise["blur sharpness"]["defocused"][low]
assert np.all(lap_defocus_low > clean_lap)
assert np.all(blur_defocus_low < clean_blur)

_fig, _axes = plt.subplots(1, 4, figsize=(12, 3.3))
for _ax, (name, series) in zip(_axes, noise.items()):
    _ax.plot(PHOTON_GRID, series["sharp"].mean(axis=1), marker="o", label="in focus", color="#0072B2")
    _ax.plot(PHOTON_GRID, series["defocused"].mean(axis=1), marker="s", label="defocus 2", color="#D55E00")
    _ax.set_xscale("log")
    _ax.set_title(name, fontsize=10)
    _ax.set_xlabel("photon scale")
_axes[0].set_ylabel("score")
_axes[-1].legend(frameon=False, fontsize=8)
_fig.tight_layout()
plt.show()
print(
    "At photon scale 15, Laplacian variance of every defocused field exceeds "
    "the clean in-focus field of the same object "
    f"(means {lap_defocus_low.mean():.3f} vs {clean_lap.mean():.3f})."
)
print(
    "Blur sharpness does not cross "
    f"(means {blur_defocus_low.mean():.3f} vs {clean_blur.mean():.3f})."
)


**What this shows.** The cell asserts two comparisons at photon scale 15, against the same objects exposed at the default 800 photons. Every defocused noisy field has a *higher* Laplacian variance than its clean in-focus twin, so a cutoff set on clean images would call the noisy defocused field the sharper one. Blur sharpness does not reverse: the defocused field stays below its clean twin. Normalized variance and the spectral slope move toward each other as the photon scale drops, but the defocused curve stays below the in-focus curve on this grid. Noise is not a universal failure. It is a failure of the score that reads noise as detail.


<a id="exp-content"></a>

## 6. Experiment 2 — How much specimen is in the field

**Question.** With defocus and exposure fixed, do the scores stay put when the number of ellipses changes? What does an empty field, which is only read noise and photon noise on a black image, receive?

**Prediction.** Normalized variance and Laplacian variance should move, because both grow when there is more edge energy. Blur sharpness should move less: it asks whether edges, if present, are already blurred. An empty field has no edges, so the two families of scores should disagree about it.


In [ ]:
# ── Experiment 2: object density, and an empty field ──
DENSITIES = np.array([8, 30, 60, 120])
N_DENSITY = 6

density = {name: np.empty((len(DENSITIES), N_DENSITY)) for name in METRICS}
for n_index, n_cells in enumerate(DENSITIES):
    for field in range(N_DENSITY):
        image = acquire(
            make_object(100 + field, n_cells=int(n_cells)),
            rng=make_rng(12, field, n_index),
        )
        for name, score in METRICS.items():
            density[name][n_index, field] = score(image)

empty_scores = {name: [] for name in METRICS}
for field in range(N_DENSITY):
    image = acquire(np.zeros((SIZE, SIZE)), rng=make_rng(12, field, 99))
    for name, score in METRICS.items():
        empty_scores[name].append(score(image))
empty_scores = {name: np.asarray(vals) for name, vals in empty_scores.items()}

nv_means = density["normalized variance"].mean(axis=1)
bs_means = density["blur sharpness"].mean(axis=1)
assert nv_means.max() / nv_means.min() > 1.15
assert bs_means.max() - bs_means.min() < 0.1
assert empty_scores["normalized variance"].mean() < 0.05
assert empty_scores["blur sharpness"].mean() > bs_means.max()

_fig, _axes = plt.subplots(1, 4, figsize=(12, 3.3))
for _ax, (name, values) in zip(_axes, density.items()):
    _ax.plot(DENSITIES, values.mean(axis=1), marker="o", color="#0072B2", label="with objects")
    _ax.axhline(empty_scores[name].mean(), color="#D55E00", ls="--", label="empty field")
    _ax.set_title(name, fontsize=10)
    _ax.set_xlabel("ellipse count")
_axes[0].set_ylabel("score")
_axes[-1].legend(frameon=False, fontsize=8)
_fig.tight_layout()
plt.show()
print(
    "Normalized variance across densities spans "
    f"{nv_means.min():.3f} to {nv_means.max():.3f}; "
    f"blur sharpness spans {bs_means.min():.3f} to {bs_means.max():.3f}."
)
print(
    "Empty-field means: "
    + ", ".join(f"{name} {vals.mean():.3f}" for name, vals in empty_scores.items())
)


**What this shows.** The cell asserts that the field-averaged normalized variance changes by more than 15% across this density range, while blur sharpness stays inside a range of 0.1. Focus did not change. Laplacian variance rises with the ellipse count because there are more edges, which is the same confound in a different score.

The empty field is the sharp disagreement. Its normalized variance is below 0.05, so a focus cutoff would reject it as blurred. Its blur sharpness is *higher* than every density mean above: with no edges, re-blurring has little left to change, and the metric reports a sharp image. An empty well is a failed acquisition. These two scores would file that failure under opposite conclusions.


<a id="exp-clip"></a>

## 7. Experiment 3 — Clipping

**Question.** If a defocused exposure is gained until highlights clip at 1, can its normalized variance exceed the in-focus exposure of the same object?

**Prediction.** Clipping flattens bright pixels and reshapes the histogram, so a ratio of variance to mean can move either way. The saturated fraction should mark the clip directly. Laplacian variance of a defocused field should stay below the in-focus field, because clipping does not put the lost high frequencies back.


In [ ]:
# ── Experiment 3: defocused and clipped, versus in focus ──
N_CLIP = 12
GAINS = np.array([1.0, 1.5, 2.5, 4.0])

clip_nv = np.empty((len(GAINS), N_CLIP))
clip_sat = np.empty((len(GAINS), N_CLIP))
focus_nv = np.empty(N_CLIP)
focus_lap = np.empty(N_CLIP)
focus_sat = np.empty(N_CLIP)
soft_lap = np.empty(N_CLIP)
examples = []

for field in range(N_CLIP):
    obj = make_object(field)
    sharp = acquire(obj, rng=make_rng(13, field, 0))
    focus_nv[field] = normalized_variance(sharp)
    focus_lap[field] = laplacian_variance(sharp)
    focus_sat[field] = saturated_fraction(sharp)
    if field == 0:
        examples.append(("in focus", sharp))
    for g_index, gain in enumerate(GAINS):
        soft = acquire(obj, defocus=2.0, gain=float(gain), rng=make_rng(13, field, g_index + 1))
        clip_nv[g_index, field] = normalized_variance(soft)
        clip_sat[g_index, field] = saturated_fraction(soft)
        if field == 0 and gain == 4.0:
            examples.append(("defocus 2, gain 4", soft))
    soft_lap[field] = laplacian_variance(
        acquire(obj, defocus=2.0, gain=4.0, rng=make_rng(13, field, 5))
    )

n_crossed = int(np.count_nonzero(clip_nv[-1] > focus_nv))
assert n_crossed >= 8
assert focus_sat.mean() < 0.02
assert clip_sat[-1].mean() > 0.1
assert np.all(soft_lap < focus_lap)

_fig, _axes = plt.subplots(1, 3, figsize=(11, 3.3))
_axes[0].plot(GAINS, clip_nv.mean(axis=1), marker="o", color="#D55E00", label="defocus 2")
_axes[0].axhline(focus_nv.mean(), color="#0072B2", ls="--", label="in focus, gain 1")
_axes[0].set_xlabel("gain before clipping")
_axes[0].set_ylabel("normalized variance")
_axes[0].legend(frameon=False, fontsize=8)
_axes[0].set_title("Score")

_axes[1].plot(GAINS, clip_sat.mean(axis=1), marker="o", color="#D55E00")
_axes[1].axhline(focus_sat.mean(), color="#0072B2", ls="--")
_axes[1].set_xlabel("gain before clipping")
_axes[1].set_ylabel("saturated fraction")
_axes[1].set_title("Gate")

for _image, _color, _label in (
    (examples[0][1].ravel(), "#0072B2", "in focus"),
    (examples[1][1].ravel(), "#D55E00", "defocus 2, gain 4"),
):
    _axes[2].hist(_image, bins=40, range=(0, 1), histtype="step", color=_color, label=_label, density=True)
_axes[2].set_xlabel("pixel value")
_axes[2].set_ylabel("density")
_axes[2].legend(frameon=False, fontsize=8)
_axes[2].set_title("Histogram, one object")
_fig.tight_layout()
plt.show()
print(
    f"Normalized variance crossed on {n_crossed} of {N_CLIP} objects "
    f"(defocused gain-4 mean {clip_nv[-1].mean():.3f}, "
    f"in-focus mean {focus_nv.mean():.3f})."
)
print(
    f"Saturated fraction {focus_sat.mean():.4f} in focus, "
    f"{clip_sat[-1].mean():.3f} after defocus and gain 4."
)
print(
    "Laplacian variance did not cross "
    f"(defocused {soft_lap.mean():.4f} < in focus {focus_lap.mean():.4f} on every object)."
)


**What this shows.** On at least 8 of the 12 objects, the cell asserts that normalized variance of the defocused, gain-4 exposure is higher than the in-focus exposure. The margin is small; the plot of means is the place to see it, and the printout gives the two means. The saturated fraction separates the same pair without ambiguity: below 0.02 in focus, above 0.1 after clipping. Laplacian variance stays lower on every defocused object. A screen that only stores normalized variance cannot tell this clipped, defocused field from a sharp one. A screen that also stores the saturated fraction can.


<a id="exp-exposure"></a>

## 8. Experiment 4 — Vignetting and exposure

**Question.** Do blur sharpness and the spectral slope stay put when an in-focus field is vignetted or dimmed, while the variance scores drop?

**Prediction.** A multiplicative shading or a global gain changes the amplitude of edges, so Laplacian variance, which scales with that amplitude, should drop. Blur sharpness is a comparison between an image and a re-blurred copy of itself, so a smooth multiplier should mostly cancel. The spectral slope is a shape in log-log coordinates and should also be close to invariant.

Two corrections are then applied, not as a recommended pipeline but as a check on the mechanism. Variance over the squared mean cancels a global rescale when noise and clipping are negligible. Dividing by a Gaussian of width $n/8$ is a rough flat-field.


In [ ]:
# ── Experiment 4: vignette and dim exposure on in-focus fields ──
N_EXPOSURE = 6


def _ratio(score, tag, **kwargs):
    """Mean of score(nuisance) / score(default) over fields."""
    ratios = []
    for field in range(N_EXPOSURE):
        obj = make_object(20 + field)
        ref = acquire(obj, rng=make_rng(14, field, tag, 0))
        alt = acquire(obj, rng=make_rng(14, field, tag, 1), **kwargs)
        ratios.append(score(alt) / score(ref))
    return float(np.mean(ratios))


lap_vig = _ratio(laplacian_variance, 1, vignette=0.9)
bs_vig = _ratio(blur_sharpness, 1, vignette=0.9)
nv_dim = _ratio(normalized_variance, 2, gain=0.25)
bs_dim = _ratio(blur_sharpness, 2, gain=0.25)
slope_vig = _ratio(spectral_slope, 1, vignette=0.9)

assert lap_vig < 0.6
assert 0.95 <= bs_vig <= 1.05
assert nv_dim < 0.45
assert bs_dim > 0.95
assert 0.9 <= slope_vig <= 1.1

print(f"Vignette 0.9: Laplacian variance ratio {lap_vig:.3f}, blur sharpness {bs_vig:.3f}, slope {slope_vig:.3f}")
print(f"Gain 0.25: normalized variance ratio {nv_dim:.3f}, blur sharpness {bs_dim:.3f}")

# Corrections. Same fields, new explicit streams.
vom_gain = []
nv_gain = []
nv_vig = []
nv_vig_ff = []
lap_vig_raw = []
lap_vig_ff = []
for field in range(N_EXPOSURE):
    obj = make_object(20 + field)
    ref = acquire(obj, rng=make_rng(15, field, 0))
    dim = acquire(obj, gain=0.25, rng=make_rng(15, field, 1))
    vig = acquire(obj, vignette=0.85, rng=make_rng(15, field, 2))
    nv_gain.append(normalized_variance(dim) / normalized_variance(ref))
    vom_gain.append(variance_over_mean_squared(dim) / variance_over_mean_squared(ref))
    nv_vig.append(normalized_variance(vig) / normalized_variance(ref))
    nv_vig_ff.append(normalized_variance(flat_field(vig)) / normalized_variance(flat_field(ref)))
    lap_vig_raw.append(laplacian_variance(vig) / laplacian_variance(ref))
    lap_vig_ff.append(laplacian_variance(flat_field(vig)) / laplacian_variance(flat_field(ref)))

nv_gain, vom_gain = np.mean(nv_gain), np.mean(vom_gain)
nv_vig, nv_vig_ff = np.mean(nv_vig), np.mean(nv_vig_ff)
lap_vig_raw, lap_vig_ff = np.mean(lap_vig_raw), np.mean(lap_vig_ff)

assert nv_gain < 0.45
assert vom_gain > 0.7
assert nv_vig < 0.75
assert nv_vig_ff > 0.85
assert lap_vig_raw < 0.6
assert lap_vig_ff > 0.9

print(
    f"Gain 0.25 ratios: variance/mean {nv_gain:.3f}, variance/mean² {vom_gain:.3f}."
)
print(
    f"Vignette 0.85, normalized variance {nv_vig:.3f} raw and {nv_vig_ff:.3f} after flat-field; "
    f"Laplacian variance {lap_vig_raw:.3f} raw and {lap_vig_ff:.3f} after flat-field."
)


**What this shows.** At vignette strength 0.9 the Laplacian variance falls below 60% of the unshaded value, while blur sharpness stays within 5% and the spectral-slope ratio stays within 10% of 1. The slope ratio is near 1 because both numbers are negative and nearly equal; the score did not become "sharper" in a meaningful way. Dimming the exposure to a gain of 0.25 drops normalized variance below 45% of the original and leaves blur sharpness above 95%.

Variance over the squared mean moves the dim-exposure ratio back above 0.7. It does not undo additive read noise, so it does not return exactly to 1. The flat-field brings the vignette's normalized-variance ratio back above 0.85 and its Laplacian-variance ratio back above 0.9. Shading was a nuisance for the variance scores because it was still in the image, not because those scores are incapable of ranking focus.


<a id="summary"></a>

## 9. Where the scores disagree

Each cell is the percent change of a score on an **in-focus** field, relative to the default exposure of the same object. A large entry means the nuisance moves the score even though focus did not change. The second table is the fraction of fields on which that in-focus nuisance scores *worse* than the same object at defocus 1 with an otherwise clean exposure. That fraction is a false blur ranking: the metric prefers a blurred field to an in-focus field that only differs by the nuisance.


In [ ]:
# ── Summary: percent change, and false ranking against defocus 1 ──
N_SUMMARY = 6
NUISANCES = [
    ("low photons", {"photons": 15}),
    ("vignette 0.9", {"vignette": 0.9}),
    ("gain 0.25", {"gain": 0.25}),
    ("gain 4, clip", {"gain": 4.0}),
    ("120 ellipses", None),
]

percent = np.empty((len(METRICS), len(NUISANCES)))
false_blur = np.empty((len(METRICS), len(NUISANCES)))
names = list(METRICS)

for n_index, (label, kwargs) in enumerate(NUISANCES):
    delta = {name: [] for name in METRICS}
    worse = {name: [] for name in METRICS}
    for field in range(N_SUMMARY):
        if label == "120 ellipses":
            obj = make_object(30 + field, n_cells=120)
            ref_obj = make_object(30 + field, n_cells=40)
        else:
            obj = make_object(30 + field)
            ref_obj = obj
        ref = acquire(ref_obj, rng=make_rng(16, field, n_index, 0))
        alt = acquire(obj, rng=make_rng(16, field, n_index, 1), **(kwargs or {}))
        blurred = acquire(ref_obj, defocus=1.0, rng=make_rng(16, field, n_index, 2))
        for name, score in METRICS.items():
            base = score(ref)
            delta[name].append(100.0 * (score(alt) - base) / abs(base))
            worse[name].append(score(alt) < score(blurred))
    for m_index, name in enumerate(names):
        percent[m_index, n_index] = np.mean(delta[name])
        false_blur[m_index, n_index] = np.mean(worse[name])

def _heatmap(ax, matrix, fmt, title):
    image = ax.imshow(matrix, cmap="coolwarm", aspect="auto")
    ax.set_xticks(range(len(NUISANCES)))
    ax.set_xticklabels([n[0] for n in NUISANCES], rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(title, fontsize=10)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            ax.text(col, row, format(matrix[row, col], fmt), ha="center", va="center", fontsize=8)
    return image

_fig, _axes = plt.subplots(1, 2, figsize=(12, 3.6))
_heatmap(_axes[0], percent, ".0f", "Percent change vs in-focus default")
_heatmap(_axes[1], false_blur, ".2f", "Fraction scored worse than defocus 1")
_fig.tight_layout()
plt.show()

# Low photons must inflate Laplacian variance; vignette must barely move blur sharpness.
lap_row = names.index("Laplacian variance")
bs_row = names.index("blur sharpness")
assert percent[lap_row, 0] > 50
assert abs(percent[bs_row, 1]) < 5
print("Percent change (rows are scores, columns are nuisances):")
print(np.array2string(percent, precision=1, suppress_small=True))


**What this shows.** The asserted cells match the earlier experiments. Low photons raise Laplacian variance by more than 50% on an in-focus field. A vignette of 0.9 changes blur sharpness by less than 5%. The right-hand table is the decision-relevant view: a nuisance that pushes an in-focus field below the defocus-1 score will be rejected by any cutoff that sits between those two. Read the printed matrix rather than a single summary number. Density and exposure do not hit every score, which is why a screen keeps more than one of them.


<a id="decision"></a>

## 10. A cutoff, with fields held out

The label is "reject" or "keep", not a continuous focus value. Kept exposures are in focus: clean, noisy (photon scale 40), vignetted (strength 0.8), or dim (gain 0.35). Rejected exposures are defocused by 2, alone or combined with that same noise, a clipping gain of 4, or the vignette.

Forty objects are split by object id: 0–19 fit the cutoff, 20–39 test it. No exposure of a test object is used to choose the cutoff. Splitting exposures at random would leak the object, the same mistake as splitting a patient's scans across train and test. The [academic-approach notebook](../../AdvancedMedicalAI/AcademicApproach/academic_approach_medical_imaging_ml.ipynb) discusses that leak; here the group is the synthetic object.

For each score, the cutoff is the value that maximizes balanced accuracy on the development objects. A field is rejected when its score is **strictly below** the cutoff. Two development sets are used:

- clean only: the clean exposure and the plain defocused exposure;
- all eight exposures.

Test false-positive rate: fraction of kept fields that are rejected. Test false-negative rate: fraction of defocused fields that are kept. The AUC uses the score itself as the ranking of "keep", so 1 means every kept test field outranks every rejected one. AUC does not depend on the cutoff.

Empty fields are scored afterwards and are not used to fit either cutoff. Experiment 2 already said the scores disagree about them.


In [ ]:
# ── Held-out cutoff ──
CONDITIONS = [
    ("clean", {}, 0),
    ("noisy", {"photons": 40}, 0),
    ("vignette", {"vignette": 0.8}, 0),
    ("dim", {"gain": 0.35}, 0),
    ("blur", {"defocus": 2.0}, 1),
    ("blur, noisy", {"defocus": 2.0, "photons": 40}, 1),
    ("blur, clipped", {"defocus": 2.0, "gain": 4.0}, 1),
    ("blur, vignette", {"defocus": 2.0, "vignette": 0.8}, 1),
]
N_OBJECTS = 40
DEV_END = 20


def cutoff_max_balanced_accuracy(scores, reject):
    """Lowest score cutoff with the best balanced accuracy.

    Reject when `score < cutoff`. Ties keep the cutoff that appears first
    among the sorted unique scores, which is the smaller one.
    """
    scores = np.asarray(scores, dtype=float)
    reject = np.asarray(reject, dtype=bool)
    best = None
    for cutoff in np.unique(scores):
        predicted = scores < cutoff
        tp = np.count_nonzero(predicted & reject)
        fn = np.count_nonzero(~predicted & reject)
        tn = np.count_nonzero(~predicted & ~reject)
        fp = np.count_nonzero(predicted & ~reject)
        balanced = 0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))
        if best is None or balanced > best[0]:
            best = (balanced, float(cutoff))
    return best[1], best[0]


rows = []
for field in range(N_OBJECTS):
    obj = make_object(200 + field)
    for c_index, (name, kwargs, reject) in enumerate(CONDITIONS):
        image = acquire(obj, rng=make_rng(20, field, c_index), **kwargs)
        row = {"field": field, "condition": name, "reject": reject}
        for metric_name, score in METRICS.items():
            row[metric_name] = score(image)
        rows.append(row)

dev = np.array([row["field"] < DEV_END for row in rows])
test = ~dev
condition_names = [name for name, _, _ in CONDITIONS]


def rates_for(metric, cutoff):
    """Overall and per-condition error rates on the test objects."""
    scores = np.array([row[metric] for row in rows])
    reject = np.array([row["reject"] for row in rows], dtype=bool)
    predicted = scores < cutoff
    keep = ~reject
    fpr = np.count_nonzero(predicted[test] & keep[test]) / np.count_nonzero(keep[test])
    fnr = np.count_nonzero(~predicted[test] & reject[test]) / np.count_nonzero(reject[test])
    auc = float(roc_auc_score(~reject[test], scores[test]))
    by_condition = {}
    for name in condition_names:
        selected = np.array([row["condition"] == name for row in rows]) & test
        is_reject = CONDITIONS[condition_names.index(name)][2] == 1
        if is_reject:
            by_condition[name] = float(np.mean(scores[selected] >= cutoff))
        else:
            by_condition[name] = float(np.mean(scores[selected] < cutoff))
    return {"fpr": float(fpr), "fnr": float(fnr), "auc": auc, "by_condition": by_condition}


results = {}
for metric in METRICS:
    scores = np.array([row[metric] for row in rows])
    reject = np.array([row["reject"] for row in rows], dtype=bool)
    clean_pair = np.array([row["condition"] in ("clean", "blur") for row in rows]) & dev
    cutoff_clean, bal_clean = cutoff_max_balanced_accuracy(scores[clean_pair], reject[clean_pair])
    cutoff_all, bal_all = cutoff_max_balanced_accuracy(scores[dev], reject[dev])
    results[metric] = {
        "clean": {"cutoff": cutoff_clean, "balanced": bal_clean, **rates_for(metric, cutoff_clean)},
        "all": {"cutoff": cutoff_all, "balanced": bal_all, **rates_for(metric, cutoff_all)},
    }
    print(
        f"{metric}: clean-only cutoff {cutoff_clean:.4g} "
        f"(dev balanced accuracy {bal_clean:.2f}), "
        f"test FPR {results[metric]['clean']['fpr']:.2f}, "
        f"FNR {results[metric]['clean']['fnr']:.2f}, "
        f"AUC {results[metric]['clean']['auc']:.2f}"
    )
    print(
        "    per condition (FPR if kept, FNR if rejected): "
        + ", ".join(
            f"{name} {results[metric]['clean']['by_condition'][name]:.2f}"
            for name in condition_names
        )
    )
    print(
        f"    all-exposure cutoff {cutoff_all:.4g}, "
        f"test FPR {results[metric]['all']['fpr']:.2f}, "
        f"FNR {results[metric]['all']['fnr']:.2f}"
    )
    print(
        "    all-exposure per condition: "
        + ", ".join(
            f"{name} {results[metric]['all']['by_condition'][name]:.2f}"
            for name in condition_names
        )
    )

nv_clean = results["normalized variance"]["clean"]["by_condition"]
assert nv_clean["vignette"] == 1.0
assert nv_clean["dim"] == 1.0
assert nv_clean["blur, clipped"] == 1.0
bs_clean = results["blur sharpness"]["clean"]["by_condition"]
assert bs_clean["vignette"] == 0.0
assert bs_clean["dim"] == 0.0
assert bs_clean["blur"] == 0.0
assert bs_clean["blur, noisy"] == 0.0
assert bs_clean["blur, clipped"] == 0.0
assert bs_clean["blur, vignette"] == 0.0
assert results["blur sharpness"]["clean"]["fnr"] == 0.0
assert results["blur sharpness"]["clean"]["fpr"] <= 0.05
lap_clean = results["Laplacian variance"]["clean"]
lap_all = results["Laplacian variance"]["all"]
assert lap_all["fnr"] > lap_clean["fnr"]
assert lap_all["by_condition"]["blur, clipped"] > lap_clean["by_condition"]["blur, clipped"]
assert lap_all["by_condition"]["vignette"] < lap_clean["by_condition"]["vignette"]
assert lap_all["by_condition"]["blur"] == 0.0

# Empty fields of the test objects. Not used to choose cutoffs.
empty_reject = {name: [] for name in METRICS}
for field in range(DEV_END, N_OBJECTS):
    image = acquire(np.zeros((SIZE, SIZE)), rng=make_rng(21, field))
    for name, score in METRICS.items():
        cutoff = results[name]["clean"]["cutoff"]
        empty_reject[name].append(score(image) < cutoff)
empty_rate = {name: float(np.mean(flags)) for name, flags in empty_reject.items()}
print("Fraction of empty test fields rejected by the clean-only cutoff:")
print(", ".join(f"{name} {rate:.2f}" for name, rate in empty_rate.items()))
assert empty_rate["normalized variance"] == 1.0
assert empty_rate["blur sharpness"] == 0.0

_fig, _axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
x = np.arange(len(condition_names))
width = 0.18
for _ax, rule in zip(_axes, ("clean", "all")):
    for offset, name in enumerate(METRICS):
        vals = [results[name][rule]["by_condition"][c] for c in condition_names]
        _ax.bar(x + (offset - 1.5) * width, vals, width=width, label=name)
    _ax.set_xticks(x)
    _ax.set_xticklabels(condition_names, rotation=30, ha="right", fontsize=8)
    _ax.set_ylim(0, 1.05)
    _ax.set_title("Cutoff fit on " + ("clean vs defocus only" if rule == "clean" else "all eight exposures"))
_axes[0].set_ylabel("FPR on kept conditions, FNR on rejected ones")
_axes[1].legend(frameon=False, fontsize=8, loc="upper right")
_fig.tight_layout()
plt.show()


**What this shows.** The assertions lock the pattern that the bars should show.

With a cutoff fit only on clean versus defocused development fields, normalized variance rejects every vignetted test field and every dim test field (false-positive rate 1 on those two conditions) and keeps every clipped defocused test field (false-negative rate 1). Those in-focus fields are below the cutoff because Experiment 4 moved the variance, and the clipped defocused fields are above it because Experiment 3 moved it back. Blur sharpness does not make those mistakes: every vignetted, dim, and defocused test field is classified correctly. Its false-negative rate is 0. Its false-positive rate is at most 0.05, and any such error is a clean field sitting next to the cutoff, which you can see in the per-condition line. The spectral slope's rates are in the same printout.

Fitting the Laplacian cutoff on all eight exposures trades one error for another. The cell asserts that this cutoff rejects fewer vignetted in-focus fields than the clean-only cutoff, and that it keeps more of the clipped defocused fields, so the overall false-negative rate rises. Plain blur stays rejected. A single cutoff on an overlapping score cannot remove every nuisance at once.

Empty test fields, which never entered the fit, are all rejected by normalized variance and none of them by blur sharpness. One score flags an empty well as out of focus. The other accepts it as sharp. Neither statement is a substitute for noticing that the well is empty.


<a id="takeaways"></a>

## 11. Takeaways

- On a fixed synthetic object and a clean exposure, all four scores fall with defocus (Spearman ≤ −0.9 on the field average). That is the only regime in which they agree.
- Laplacian variance treats a shortage of photons as extra sharpness. At photon scale 15, every defocused field outscored the clean in-focus field of the same object.
- Normalized variance and Laplacian variance change with the number of ellipses. Blur sharpness, within 0.1 across that range, does not. An empty field gets a very low normalized variance and a blur-sharpness score above the fields that contain objects.
- Clipping can lift the normalized variance of a defocused exposure above the in-focus exposure. The fraction of pixels at 0.99 or above shows the clip; Laplacian variance does not reverse.
- A vignette and a dim gain move the variance scores and leave blur sharpness and the mid-band slope nearly still. Dividing by a smooth illumination estimate puts most of that vignette shift back.
- A cutoff fit on clean fields inherits whichever nuisance the score is sensitive to. On the held-out objects, normalized variance rejects every dim and vignetted keep-field and accepts every clipped defocused field. Blur sharpness keeps those vignetted and dim fields and rejects the defocused ones; it still accepts the empty fields. Widening the Laplacian fit stops flagging the vignette and starts keeping the clipped defocused fields.


<a id="exercises"></a>

## 12. Questions and exercises

1. In `spectral_slope`, the fit ends at $k = n/8$. Change the upper limit to $n/4$, rerun Experiment 0, and explain the new Spearman correlation. Which part of the spectrum is the extra band measuring once the object is strongly blurred?
2. Before running any code: a 5×5 block of saturated pixels is written onto an otherwise in-focus field. Which score do you expect to move most, and why? Then add the block inside `acquire` and check.
3. Using the development objects only, reject a field if its saturated fraction exceeds 0.05 **or** its normalized variance is below the clean-only cutoff. On the test objects, recount false positives and false negatives by condition. Which error from section 10 does the second gate remove, and which errors does it leave?
4. Recompute normalized variance on pixels above 0.05 only, and again as the mean of the score on non-overlapping 32×32 tiles. Does either version reduce the density effect in Experiment 2 or the empty-field disagreement?
5. In `acquire`, replace the Gaussian of width $\sigma$ with `scipy.ndimage.uniform_filter` of a growing size. Does Experiment 0 still pass for blur sharpness? The re-blur inside `blur_effect` is itself a uniform filter, so think about what a uniform-filter defocus does to that comparison.

### Bonus

Name one physical effect of a real widefield fluorescence exposure that this generator cannot produce, and say which score you would trust least until that effect is in the simulation. Depth-dependent blur, bleed from a second channel, and stage drift are all admissible answers; the point is to connect the effect to a score's assumption.


<a id="references"></a>

## 13. References

The sharpness functions that are not library calls are teaching implementations. They are not claimed to match CellProfiler's numeric output, and no CellProfiler source, pipeline, or tutorial text is included here.

1. Crete, F., Dolmiere, T., Ladret, P., and Nicolas, M. (2007). The blur effect: perception and estimation with a new no-reference perceptual blur metric. *Proceedings of SPIE* 6492, 64920I. https://doi.org/10.1117/12.702790. Implemented in scikit-image as `skimage.measure.blur_effect`.
2. Sun, Y., Duthaler, S., and Nelson, B. J. (2004). Autofocusing in computer microscopy: selecting the optimal focus algorithm. *Microscopy Research and Technique* 65(3), 139–149. https://doi.org/10.1002/jemt.20118. The normalized-variance comparison cited above.
3. Pech-Pacheco, J. L., Cristobal, G., Chamorro-Martinez, J., and Fernandez-Valdivia, J. (2000). Diatom autofocusing in brightfield microscopy: a comparative study. *Proceedings of the 15th International Conference on Pattern Recognition*, vol. 3, 314–317. https://doi.org/10.1109/ICPR.2000.903548. Related Laplacian focus score; not the function implemented here.
4. Field, D. J. (1987). Relations between the statistics of natural images and the response properties of cortical cells. *Journal of the Optical Society of America A* 4(12), 2379–2394. https://doi.org/10.1364/josaa.4.002379. Background for looking at a log-log spectrum. Fluorescence fields are not natural images, and the fit in this notebook is not Field's measurement.
5. Bray, M.-A., and Carpenter, A. E. (2018). Quality control for high-throughput imaging experiments using machine learning in CellProfiler. *Methods in Molecular Biology* 1683, 89–112. https://doi.org/10.1007/978-1-4939-7357-6_7.
6. CellProfiler manual, *MeasureImageQuality*. https://cellprofiler-manual.s3.amazonaws.com/CPmanual/MeasureImageQuality.html. Names the screen measurements (focus score, power log-log slope, percent maximal) that this notebook studies in reduced form.
7. van der Walt, S., Schönberger, J. L., Nunez-Iglesias, J., Boulogne, F., Warner, J. D., Yager, N., Gouillart, E., Yu, T., and the scikit-image contributors (2014). scikit-image: image processing in Python. *PeerJ* 2, e453. https://doi.org/10.7717/peerj.453.
